# FLAG 2027 - Notebook 26: stage 2 of NB24 - Whisper-v3 encoder features for VoxCeleb2 + MAV language table (for NAP)

NB24 result (29 Oct): R4 + Whisper-large-v3 encoder (layers 20-28) + NAP-8 = CV 25.15 vs 25.53 (gender 31.28 vs 32.15). It is the only lever left with a consistent positive signal. To put it into the final system (NB20 system **C**) we need:

1. Whisper features for the SAME VoxCeleb2 clips as NB07/13/18 (re-download the two zips, rebuild, alignment check on ECAPA, extract) -> `features/vox_<tag>_whisper.npy`
2. The language of every MAV training row (English / Urdu / Hindi / German) in our row order -> `features/mav_lang.npz`. NB20 v4 builds the NAP basis from it (training data only; no test data touched).

About 2.5-3 h (download ~40 min, ffmpeg ~20 min, Whisper ~1.5 h). Resumable per zip. Needs the HF token in Colab Secrets as `HF_TOKEN` (VoxCeleb mirror is gated). Paste the **SUMMARY**.

In [ ]:
# Cell 1 - Drive, GPU, packages, HF token (from Colab Secrets; never paste it anywhere)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again. Do NOT continue on CPU (Whisper would take weeks).'
dev = 'cuda'
HF_TOKEN = userdata.get('HF_TOKEN')
print('[v2 29-Oct] Device:', torch.cuda.get_device_name(0), '| HF token found:', HF_TOKEN is not None, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - paths + encoders: ECAPA (alignment check), language-ID (same filter as NB07), Whisper-v3 encoder (identical to NB24 Cell 3)
import os, re, json, time, zipfile, subprocess
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max()); b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def enc_ecapa(paths, bs=16):
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa (check)'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def enc_whisper(paths, bs=8, layers=(20, 29)):
    from transformers import WhisperFeatureExtractor, WhisperModel
    name = 'openai/whisper-large-v3'
    fe = WhisperFeatureExtractor.from_pretrained(name); enc = WhisperModel.from_pretrained(name, torch_dtype=torch.float16).encoder.to(dev).eval(); out = []
    for i in tqdm(range(0, len(paths), bs), desc='whisper-v3 enc'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * 30] for p in paths[i:i + bs]]
        x = fe(wavs, sampling_rate=16000, return_tensors='pt').input_features.to(dev, torch.float16)
        with torch.no_grad():
            hs = enc(x, output_hidden_states=True).hidden_states
        h = torch.stack(hs[layers[0]:layers[1]]).float().mean(0)
        for k_, w in enumerate(wavs):
            n = max(1, min(1500, len(w) // 320)); out.append(h[k_, :n].mean(0).cpu().numpy())
    del enc; torch.cuda.empty_cache(); return np.stack(out)
def ffmpeg_audio(job):
    mp4, wav, jpg = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', jpg]).returncode
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(jpg)
# consistency check against the challenge-side Whisper features (NB24): the same encoder on 50 cached-feature rows must agree
W24 = np.load(os.path.join(FEATDIR, 'whisper.npz'))
assert 'train' in W24.files and W24['train'].shape[1] == 1280, 'features/whisper.npz from NB24 not found or wrong shape - tell Claude'
print('encoders ready | NB24 whisper features found: train %d x %d' % W24['train'].shape)

In [ ]:
# Cell 3 - rebuild the SAME VoxCeleb2 clips (NB07 selection, verified in NB18) and extract Whisper (RESUMABLE per zip)
ZIPS = [('vox2/vox2_test_mp4.zip', 40), ('vox2/vox2_mp4_1.zip', 20)]
for zname, cpi in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    out_w = os.path.join(FEATDIR, 'vox_' + tag + '_whisper.npy')
    if os.path.exists(out_w):
        print(tag, 'already done'); continue
    OLD = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    t0 = time.time()
    zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    zf = zipfile.ZipFile(zpath); tree = {}
    for n in zf.namelist():
        mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
        if mm:
            tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
    rng = np.random.default_rng(0); picked = []
    for pid, vids in tree.items():
        pools = [list(rng.permutation(v)) for v in vids.values()]; chosen = []
        while len(chosen) < cpi and any(pools):
            for p in pools:
                if p and len(chosen) < cpi:
                    chosen.append(p.pop())
        picked += [(pid, c) for c in chosen]
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True); jobs = []
    for i, (pid, member) in enumerate(picked):
        jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i), '%s/%06d.jpg' % (W, i)))
    zf.close(); os.remove(zpath)
    with ThreadPoolExecutor(8) as ex:
        ok = list(tqdm(ex.map(ffmpeg_audio, jobs), total=len(jobs), desc='ffmpeg'))
    keep = [i for i, g in enumerate(ok) if g]
    wavs = [jobs[i][1] for i in keep]; spk = np.array(['vox_' + picked[i][0] for i in keep])
    lang = language_of(wavs); nb = lang != 'bn'
    wavs = [w for w, k in zip(wavs, nb) if k]; spk = spk[nb]
    assert len(spk) == len(OLD['spk']) and (spk == OLD['spk']).all(), 'row order differs from Notebook 07 - tell Claude'
    probe = np.random.default_rng(1).choice(len(wavs), 300, replace=False)
    e_new = enc_ecapa([wavs[j] for j in probe]); e_old = OLD['ecapa'][probe]
    cos = (e_new * e_old).sum(1) / (np.linalg.norm(e_new, axis=1) * np.linalg.norm(e_old, axis=1) + 1e-9)
    note('%s: %d rows | alignment median cosine %.4f' % (tag, len(spk), np.median(cos)))
    assert np.median(cos) > 0.98, 'rows do not line up - stop and tell Claude'
    E = enc_whisper(wavs).astype(np.float32); assert E.shape == (len(wavs), 1280)
    np.save(out_w, E); note('%s: Whisper done, %d x %d (%.0f min)' % (tag, E.shape[0], E.shape[1], (time.time() - t0) / 60))
    shutil.rmtree(W, ignore_errors=True)
print('[v1 29-Oct] VoxCeleb Whisper features:', [os.path.basename(p) for p in sorted(__import__('glob').glob(os.path.join(FEATDIR, 'vox_*_whisper.npy')))])

In [ ]:
# Cell 4 - language of every MAV training row in OUR row order (zip listings only, nothing extracted) -> features/mav_lang.npz
# Same selection logic as NB24 Cell 2 (which itself reproduces NB13's row order; verified there by the ECAPA alignment check).
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
mspk, mlang = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf); zf.close()
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            if zi.get('voices/' + t[0]) is None or zi.get('faces/' + t[1]) is None:
                continue
            mspk.append(ver + '_' + t[0].split('/')[0]); mlang.append(lf.split('_')[0])
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    zf.close()
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]            # keeps the random sequence identical to NB13/NB24
            rows.append((ver + '_' + pid, lang_))
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
xspk = [r[0] for r, k in zip(rows, keep) if k]; xlang = [r[1] for r, k in zip(rows, keep) if k]
assert xspk == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
EXT_SPK = np.array(mspk + xspk); EXT_LANG = np.array(mlang + xlang)
np.savez(os.path.join(FEATDIR, 'mav_lang.npz'), spk=EXT_SPK, lang=EXT_LANG)
n_bil = sum(len(np.unique(EXT_LANG[EXT_SPK == s])) >= 2 for s in np.unique(EXT_SPK))
note('MAV language table saved: %d rows | ' % len(EXT_SPK) + ', '.join('%s %d' % (l, (EXT_LANG == l).sum()) for l in sorted(set(EXT_LANG))) + ' | bilingual speakers %d (NB24 had 204)' % n_bil)
assert n_bil == 204, 'bilingual speaker count differs from NB24 - tell Claude'
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)